# Executive summary (full report)

**Project:** Independent review of writing-style change at Mediaite.com using stylometry, embedding-space drift, and statistical comparisons.

**What this chapter contains:** A concise reading of the per-author analysis bundles together with headline tables. Earlier chapters document collection, features, change points, drift, tests, and controls.

**Provenance:** Corpus custody and configuration fingerprints are summarized in chapter 00 and in `data/analysis/corpus_custody.json`. The closing section of this chapter lists the git revision, analysis-configuration hash, and render timestamp for this build.

## Executive summary

This section states what the bundled analysis files show for the configured author panel. It does not substitute for reading the evidence chapters or the custody record.

**Panel highlights**

1. **Embedding drift (`pb_max`).** In the current artifact set, all twelve named study authors have a positive embedding-drift summary score. The highest values among those authors include `david-gilmour` (0.598), `colby-hall` (0.589), `sarah-rumpf` (0.587), `charlie-nash` (0.537), and `joe-depaolo` (0.536). These figures are descriptive rankings from the stored results, not findings about intent or authorship.

2. **Adjusted hypothesis tests.** Under the documented per-family Benjamini–Hochberg adjustment, a subset of tests meets the pre-registered significance and effect-size criteria. Exact headline counts appear in the metrics table, which is recomputed from the same `*_result.json` inputs as the earlier chapters.

3. **Combined stylometry and drift (`colby-hall`).** For `colby-hall`, stylometric convergence and embedding drift both exceed the declared thresholds across many windows, including 170 windows where both channels register (`ab` in `passes_via`). One representative window runs from 2026-01-08 through 2026-04-08 with five feature families flagged together: `ai_markers`, `entropy`, `lexical_richness`, `readability`, and `self_similarity`.

4. **Language-model probability channel.** When token-level probability features are not present in the feature store, the probability trajectory score is zero for all authors. For this render, the primary quantitative story is therefore carried by stylometry, embedding drift, and the joint convergence index.

**Interpretive limits**

- The metrics describe shifts in measurable text features; they do not, standing alone, establish how any article was written.
- Whether this render is confirmatory or exploratory is determined by the pre-registration check in the next block.

In [ ]:
from IPython.display import Markdown, display

from forensics.config import get_settings
from forensics.preregistration import verify_preregistration

_prereg = verify_preregistration(get_settings())
_status = _prereg.status
_msg = _prereg.message.replace('|', r'\|')
_lock_path = str(_prereg.lock_path)

_label = {
    'ok': 'Confirmatory (lock matches current settings)',
    'mismatch': 'Exploratory (lock does not match current settings)',
    'missing': 'Exploratory (no filled lock on file)',
}.get(_status, f'Unknown status ({_status})')

display(Markdown(f"""
### Pre-registration verification (computed when this chapter is rendered)

| Field | Value |
|---|---|
| Status | **{_label}** |
| Lock path | `{_lock_path}` |
| Detail | {_msg} |

When the status is confirmatory, the analysis thresholds in use match the committed lock. When the status is exploratory, thresholds were not locked to this render (or the lock no longer matches the configuration); quantitative results should be read as provisional until a matching lock and confirmatory re-run are archived.
"""))


## Top-line metrics

The next code cell loads each per-author `result.json` under `data/analysis/` and recomputes the headline metrics so the table matches the bundles used elsewhere in this volume.

In [ ]:
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype+notebook_connected+png'


In [ ]:
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype+notebook_connected+png'


In [ ]:
import json

import polars as pl
from IPython.display import Markdown, display

from forensics.config import get_project_root

ANALYSIS_DIR = get_project_root() / "data" / "analysis"
SHARED_BYLINE_SLUGS = {"mediaite", "mediaite-staff"}
RESULT_FILES = [
    p
    for p in sorted(ANALYSIS_DIR.glob("*_result.json"))
    if p.name.removesuffix("_result.json") not in SHARED_BYLINE_SLUGS
]

rows: list[dict[str, object]] = []
for path in RESULT_FILES:
    slug = path.name.removesuffix("_result.json")
    payload = json.loads(path.read_text(encoding="utf-8"))
    tests = payload.get("hypothesis_tests", []) or []
    cw = payload.get("convergence_windows", []) or []
    pa_vals = [(w.get("pipeline_a_score") or 0.0) for w in cw]
    pb_vals = [(w.get("pipeline_b_score") or 0.0) for w in cw]
    ratio_vals = [(w.get("convergence_ratio") or 0.0) for w in cw]
    ab_windows = sum(1 for w in cw if "ab" in (w.get("passes_via") or []))
    rows.append(
        {
            "slug": slug,
            "n_tests": len(tests),
            "n_sig": sum(1 for t in tests if t.get("significant")),
            "pa_max": max(pa_vals) if pa_vals else 0.0,
            "pb_max": max(pb_vals) if pb_vals else 0.0,
            "ratio_max": max(ratio_vals) if ratio_vals else 0.0,
            "n_windows": len(cw),
            "n_ab_windows": ab_windows,
        }
    )

summary = pl.DataFrame(rows)

n_authors = summary.height
authors_pb_pos = int((summary["pb_max"] > 0).sum())
total_tests = int(summary["n_tests"].sum())
total_sig = int(summary["n_sig"].sum())
ratio_max_overall = float(summary["ratio_max"].max())

colby = summary.filter(pl.col("slug") == "colby-hall")
if colby.height == 1:
    colby_pa = float(colby["pa_max"][0])
    colby_pb = float(colby["pb_max"][0])
    colby_ab = int(colby["n_ab_windows"][0])
else:
    colby_pa = colby_pb = float("nan")
    colby_ab = 0

metrics_md = f"""| metric | pre-fix | post-fix |
|---|---|---|
| authors with PB_max > 0 | 1/14 | {authors_pb_pos}/{n_authors} |
| significant tests (BH-FDR) | 0/2,277 | {total_sig:,}/{total_tests:,} |
| colby-hall PB_max | 0.00 | {colby_pb:.3f} |
| colby-hall PA_max | 0.94 | {colby_pa:.2f} |
| convergence ratio max | 0.75 | {ratio_max_overall:.2f} |
| colby-hall AB-confirmed windows | n/a | {colby_ab} |
"""
display(Markdown(metrics_md))

display(
    summary.sort("pb_max", descending=True).select(
        ["slug", "pa_max", "pb_max", "ratio_max", "n_ab_windows", "n_sig", "n_tests"]
    )
)

## Finding strength classification

Each author's strongest convergence window is graded with `classify_finding_strength` from `forensics.models.report`. Hypothesis tests passed to the classifier are **window-scoped** — only tests whose `feature_name` appears in the window's `features_converging` list count toward the tally. This prevents an author from being credited for significant tests that don't actually support the convergence window in question.

A **strong significant test** is one with corrected p < 0.01 *and* |Cohen's d| ≥ 0.8.

- **STRONG (red):** ≥3 strong significant tests *and* controls clean (`editorial_vs_author_signal > 0.7`) *and* (Pipeline C ok when probability features are available).
- **MODERATE (orange):** ≥3 significant tests *and* ≥1 strong significant test. The `≥1 strong` floor distinguishes meaningful from marginal evidence; the `≥3 sig` count keeps the bar above the trivial 2-test threshold that previously bucketed authors with vastly different evidence strength into the same tier.
- **WEAK (yellow):** ≥1 significant test.
- **NONE (green):** no significant tests in this window's features.

Probability features are unavailable in this run, so the Pipeline C clause is a no-op. The `editorial_vs_author_signal` is looked up per-target from `data/analysis/comparison_report.json` (`targets.<slug>.editorial_vs_author_signal`); controls have no such signal and default to 0.0, which structurally reserves STRONG for designated targets.

**Threshold provenance caveat:** the MODERATE bar (`≥3 sig AND ≥1 strong`) was chosen after observing that the prior `≥2 sig` bar bucketed authors with up to a 50× range in evidence quality together. This is exploratory; the bar must be locked in `data/preregistration/` before any confirmatory claim is made.

### §11.3.1 Direction concordance and volume context (exploratory)

The table in the next cell adds two **diagnostic** columns alongside `FindingStrength`:

- **`direction`** (`direction_ai`, `direction_mixed`, `direction_non_ai`, `direction_na`): after collapsing to one hypothesis test per `feature_name` (largest |Cohen's d|), each feature with a documented AI-typical direction in `AI_TYPICAL_DIRECTION` is scored as matching or opposing that prior. The **≥50%** rule (among features that have a non-null prior) labels the window `direction_ai` when at least half of those comparisons match; `direction_mixed` when some but fewer than half match; `direction_non_ai` when none match but at least one prior existed; `direction_na` when no feature had a usable prior. **This cutoff is exploratory** until it is locked in `data/preregistration/preregistration_lock.json`.

- **`volume_flag`** (`volume_stable`, `volume_growth`, `volume_ramp`, `volume_decline`, `volume_unknown`): uses `n_post / n_pre` from the first usable row in the window-scoped test list. **A ratio above 5×** is flagged as `volume_ramp` because a large baseline-to-window article-count increase is a common confound for stylometric shifts that are not specific to LLM assistance. **That threshold is exploratory** and must be pre-registered before any confirmatory claim.

**Apr 27 2026 qualitative contrast (illustrative, not a separate statistical test):** for the designated target window discussed in the review, effects on measured features with priors pointed in the AI-typical direction with a **declining** article-volume ratio (~0.44×). Several comparison authors graded MODERATE on strength alone showed **opposing** directions on most prior-scored features together with **large** volume ramps (on the order of 12×–276×). The strength tier alone does not distinguish those patterns; these columns make the contrast visible.

Read **§11.1** caveats on correlation, multiple testing, and confounding before interpreting any row. Nothing in this subsection overrides the exploratory status of the diagnostics or the pre-registration gate described above.


In [ ]:
import json

import polars as pl
from IPython.display import display

from forensics.config import get_project_root
from forensics.models.analysis import ConvergenceWindow, HypothesisTest
from forensics.models.report import (
    classify_direction_concordance,
    classify_finding_strength,
    compute_volume_ramp_flag,
)

ANALYSIS_DIR = get_project_root() / "data" / "analysis"
comparison_path = ANALYSIS_DIR / "comparison_report.json"
if comparison_path.is_file():
    comparison_blob = json.loads(comparison_path.read_text(encoding="utf-8"))
else:
    comparison_blob = {"targets": {}}

_STRENGTH_RANK = {"strong": 0, "moderate": 1, "weak": 2, "none": 3}
_DIRECTION_RANK = {
    "direction_ai": 0,
    "direction_mixed": 1,
    "direction_non_ai": 2,
    "direction_na": 3,
}
_VOLUME_RANK = {
    "volume_stable": 0,
    "volume_decline": 1,
    "volume_growth": 2,
    "volume_unknown": 3,
    "volume_ramp": 4,
}

_DISPLAY_COLS = [
    "slug",
    "strength",
    "direction",
    "volume_flag",
    "pa",
    "pb",
    "window_start",
    "window_end",
    "n_families",
    "dir_match",
    "dir_oppose",
    "n_sig",
    "volume_ratio",
]

strength_rows: list[dict[str, object]] = []
SHARED_BYLINE_SLUGS = {"mediaite", "mediaite-staff"}
for path in sorted(ANALYSIS_DIR.glob("*_result.json")):
    if path.name.removesuffix("_result.json") in SHARED_BYLINE_SLUGS:
        continue
    slug = path.name.removesuffix("_result.json")
    payload = json.loads(path.read_text(encoding="utf-8"))
    raw_windows = payload.get("convergence_windows", []) or []
    if not raw_windows:
        strength_rows.append(
            {
                "slug": slug,
                "strength": "none",
                "direction": "direction_na",
                "volume_flag": "volume_unknown",
                "volume_ratio": None,
                "pa": 0.0,
                "pb": 0.0,
                "window_start": None,
                "window_end": None,
                "n_families": None,
                "dir_match": 0,
                "dir_oppose": 0,
                "dir_no_prior": 0,
                "n_sig": 0,
            }
        )
        continue
    windows = [ConvergenceWindow.model_validate(w) for w in raw_windows]
    best = max(
        windows,
        key=lambda w: ((w.pipeline_a_score or 0.0) + (w.pipeline_b_score or 0.0)),
    )
    tests = [HypothesisTest.model_validate(t) for t in payload.get("hypothesis_tests", []) or []]
    window_features = set(best.features_converging or [])
    window_tests = (
        [t for t in tests if t.feature_name in window_features] if window_features else tests
    )
    target_block = comparison_blob.get("targets", {}).get(slug, {})
    control_comparison = {
        "editorial_vs_author_signal": target_block.get("editorial_vs_author_signal", 0.0),
    }
    strength = classify_finding_strength(
        convergence_window=best,
        hypothesis_tests=window_tests,
        control_comparison=control_comparison,
        probability_features_available=False,
    )
    direction, breakdown = classify_direction_concordance(best, window_tests)
    volume_flag, volume_ratio = compute_volume_ramp_flag(window_tests)
    strength_rows.append(
        {
            "slug": slug,
            "strength": str(strength),
            "direction": str(direction),
            "volume_flag": str(volume_flag),
            "pa": round(best.pipeline_a_score or 0.0, 3),
            "pb": round(best.pipeline_b_score or 0.0, 3),
            "window_start": best.start_date.isoformat() if best.start_date else None,
            "window_end": best.end_date.isoformat() if best.end_date else None,
            "n_families": len(best.families_converging),
            "dir_match": breakdown.n_match,
            "dir_oppose": breakdown.n_oppose,
            "dir_no_prior": breakdown.n_no_prior,
            "n_sig": sum(1 for t in window_tests if t.significant),
            "volume_ratio": round(volume_ratio, 2) if volume_ratio is not None else None,
        }
    )

strength_rows.sort(
    key=lambda r: (
        _STRENGTH_RANK.get(str(r["strength"]), 99),
        _DIRECTION_RANK.get(str(r["direction"]), 99),
        _VOLUME_RANK.get(str(r["volume_flag"]), 99),
        -float(r["pb"] if r.get("pb") is not None else 0.0),
    )
)
strength_df = pl.DataFrame(strength_rows)
display(strength_df.select(_DISPLAY_COLS))

## Limitations and scope

- Outcomes depend on the corpus snapshot, exclusion rules, and the analysis thresholds recorded in configuration and, when used, the pre-registration lock.
- The language-model probability layer contributes only when probability features are present in the feature store; otherwise the composite score reflects stylometry, embedding drift, and joint convergence.
- Natural-author controls and pooling rules follow the methodology described in project documentation.

## Recommendations

- **Lock the direction-priors registry and the 5× volume-ramp threshold in pre-registration before any external publication.** The Phase 17 diagnostic columns (`direction`, `volume_flag`) in §11.3 are exploratory until the priors and thresholds are committed in `data/preregistration/preregistration_lock.json`.

## Further documentation

- Pre-registration and threshold history: `docs/pre_registration.md` and `data/preregistration/`.
- Pipeline design and data contracts: `docs/ARCHITECTURE.md`.
- Operator commands and environment notes: `docs/RUNBOOK.md`.
- Custody record: `data/analysis/corpus_custody.json`.

## Provenance

This cell prints the git SHA and the deterministic hash of `settings.analysis` so the report is auditable downstream.

In [ ]:
import subprocess
import sys
from datetime import UTC, datetime

from IPython.display import Markdown, display

from forensics.config import get_project_root, get_settings
from forensics.utils.provenance import compute_model_config_hash

settings = get_settings()
root = get_project_root()

try:
    git_sha = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=root, stderr=subprocess.STDOUT
    ).decode().strip()
except (subprocess.CalledProcessError, FileNotFoundError) as exc:
    git_sha = f"<unavailable: {exc}>"

analysis_hash = compute_model_config_hash(settings.analysis)
run_timestamp = datetime.now(UTC).isoformat()

display(
    Markdown(
        f"""| Key | Value |
|---|---|
| Git SHA | `{git_sha}` |
| Analysis-config hash | `{analysis_hash}` |
| Report rendered at | `{run_timestamp}` |
| Python | `{sys.version.split()[0]}` |
"""
    )
)